# 06 · Do modelo à decisão — e ao produto

O enunciado pede *"informações relevantes que possam **agregar algo de valor**"*.
Este notebook é a resposta: o instrumento aplicável e o orçamento que ele implica.

> Documentos: [`docs/16`](../docs/16-trilhaC-escore-decisao-equidade.md) e [`docs/17`](../docs/17-produto-calculadora.md)


In [1]:
import sys, json
from pathlib import Path

# a raiz e onde existe src/ — funciona rodando de notebooks/ ou da raiz do repo
RAIZ = Path.cwd()
if not (RAIZ / "src").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np, pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

GOLD = RAIZ / "data" / "processed" / "gold"
def ler(nome, base=GOLD):
    return json.loads((base / nome).read_text(encoding="utf-8"))


## O escore de 5 perguntas


In [2]:
e = ler("_trilhaC_escore.json")
b = e["escores"]["B_sem_proxy_acesso"]
for var, pontos in b["tabela"]["pontos"].items():
    ordenado = sorted(pontos.items(), key=lambda x: x[1])
    print(f"{var:14} " + "   ".join(f"{k}={v:+d}" for k, v in ordenado))


idade          <35=+0   35-44=+8   45-54=+12   55-64=+15   65+=+17
imc            <25=+0   25-29=+3   30-34=+5   35+=+9
saude          excelente/muito boa=+0   boa=+6   regular/ruim=+11
hipertensao    nao=+0   sim=+7
sexo           feminino=+0   masculino=+1


In [3]:
pd.DataFrame(b["calibracao"]["faixas"])[["pontos_min", "pontos_max", "risco_%"]]


,pontos_min,pontos_max,risco_%
0,0,9,0.54
1,10,15,1.76
2,16,18,3.42
3,19,23,6.54
4,24,27,12.94
5,28,30,18.51
6,31,35,28.56
7,36,45,45.49


Da faixa mais baixa à mais alta, o risco multiplica por **84×**.


## A decisão que define o escore


In [4]:
pd.DataFrame({
    "A · com colesterol (6 perguntas)": e["escores"]["A_completo"]["metricas"],
    "B · sem proxy de acesso (5)":      e["escores"]["B_sem_proxy_acesso"]["metricas"],
    "FINDRISC aproximado":              e["findrisc"],
}).T[["roc_auc", "pr_auc", "n_variaveis", "n_avaliacao"]]


,roc_auc,pr_auc,n_variaveis,n_avaliacao
A · com colesterol (6 perguntas),0.8082,0.3671,6.0,62294.0
B · sem proxy de acesso (5),0.804,0.3595,5.0,62294.0
FINDRISC aproximado,0.7663,0.3195,NaN,62294


> **Adotamos o B.** "Você tem colesterol alto?" só é respondível por quem **já fez
> um exame de sangue** — e o notebook 05 mostrou que os prováveis não diagnosticados
> são justamente os que **não fizeram**.
>
> Um escore que exige exame prévio **não alcança quem mais precisa dele**.
> Custa 2,07% de PR-AUC. É barato.

E as 5 perguntas ainda **batem o FINDRISC** em +37,7 milésimos de ROC-AUC — na
mesma amostra de 62.294 pessoas.


## Quantos testar, quantos achar, a que custo


In [5]:
d = ler("_trilhaC_decisao.json")
cob = pd.DataFrame(d["candidatos"]["escore_5_perguntas"]["cobertura"])
cob["custo médio R$"] = cob["custo_por_caso_R$"].apply(lambda x: x[1])
cob[["%_testado", "%_casos_encontrados", "nns_acumulado", "custo médio R$"]].head(7)


,%_testado,%_casos_encontrados,nns_acumulado,custo médio R$
0,5.0,21.04,2.23,71.0
1,10.0,40.25,2.33,75.0
2,15.0,54.36,2.59,83.0
3,20.0,65.41,2.87,92.0
4,25.0,73.61,3.19,102.0
5,30.0,79.22,3.55,114.0
6,40.0,88.44,4.24,136.0


In [6]:
dec = pd.DataFrame(d["candidatos"]["escore_5_perguntas"]["por_decil"])
dec["custo médio R$"] = dec["custo_por_caso_R$"].apply(lambda x: x[1])
dec[["faixa", "risco_%", "nns", "custo médio R$", "%_dos_casos_totais"]]


,faixa,risco_%,nns,custo médio R$,%_dos_casos_totais
0,1,0.89,112.3,3593.0,3.41
1,2,4.05,24.7,790.0,3.99
2,3,5.31,18.8,603.0,6.52
3,4,12.49,8.0,256.0,12.83
4,5,18.81,5.3,170.0,12.77
5,6,29.31,3.4,109.0,25.91
6,7,46.52,2.1,69.0,34.57


**As duas faixas superiores concentram 60,5% dos casos a R$ 69–109 por caso.**
A faixa mais baixa custa **R$ 3.593** — **52× mais**.

Rastrear por ordem de escore não é refinamento: é a diferença entre um programa
viável e um inviável.


## Equidade — e o problema de auditar um rótulo enviesado


In [7]:
eq = ler("_trilhaC_equidade.json")
pd.DataFrame(eq["observado"]["raca"]["por_grupo"])[
    ["grupo", "prevalencia_%", "taxa_selecao_%", "recall_tpr",
     "precisao_ppv", "calibracao_desvio_pp"]]


,grupo,prevalencia_%,taxa_selecao_%,recall_tpr,precisao_ppv,calibracao_desvio_pp
0,branco nao-hispanico,9.96,10.57,0.4815,0.4536,-0.32
1,hispanico,10.74,13.67,0.6002,0.4714,0.02
2,multirracial nao-hispanico,8.89,16.00,0.6111,0.3398,2.34
3,negro nao-hispanico,14.24,19.26,0.6592,0.4876,-0.35
4,outro nao-hispanico,9.15,7.86,0.4078,0.4744,-1.45


In [8]:
comp = pd.DataFrame({
    eixo: {"observado": eq["observado"][eixo]["disparidade"]["amplitude_igualdade_oportunidade"],
           "corrigido pelo PU": eq["corrigido_pu"][eixo]["disparidade"]["amplitude_igualdade_oportunidade"]}
    for eixo in ("raca", "sexo", "renda", "idade")}).T
comp["direção"] = np.where(comp["corrigido pelo PU"] < comp["observado"], "melhora", "PIORA")
comp


,observado,corrigido pelo PU,direção
raca,0.2514,0.2274,melhora
sexo,0.0088,0.0062,melhora
renda,0.2750,0.2935,PIORA
idade,0.3501,0.3703,PIORA


> Corrigir pelo subdiagnóstico **muda a leitura da equidade — e não na mesma
> direção em todos os eixos**. Em raça, parte da disparidade aparente era artefato
> do rótulo. Em renda e idade, a disparidade **real é maior** que a medida.
>
> Uma auditoria de justiça que ignora o viés de verificação **subestima a
> injustiça exatamente onde ela é pior**.


## O produto


In [9]:
prod = json.loads((RAIZ / "reports/produto/modelo.json").read_text(encoding="utf-8"))
print(f"ROC-AUC              {prod['metricas']['roc_auc']}")
print(f"PR-AUC               {prod['metricas']['pr_auc']}")
print(f"termos do EBM        {len(prod['ebm']['termos'])}")
print(f"paridade Python↔JS   erro máximo {prod['paridade_export']['erro_max']:.2e}")
print(f"tamanho do JSON      {(RAIZ/'reports/produto/modelo.json').stat().st_size/1024:.0f} KB")
print(f"tamanho da página    {(RAIZ/'reports/produto/index.html').stat().st_size/1024:.0f} KB")


ROC-AUC              0.8421
PR-AUC               0.446
termos do EBM        20
paridade Python↔JS   erro máximo 1.11e-16
tamanho do JSON      42 KB
tamanho da página    59 KB


### 👉 Abra `reports/produto/index.html`

O EBM é **aditivo**, então exportamos as tabelas de consulta e a predição roda em
JavaScript com o **mesmo número** do Python — erro máximo 1,1 × 10⁻¹⁶, verificado
em 500 casos a cada build.

Um HTML de 59 KB, offline, que estima o risco, mostra **o que pesa** em cada
resposta, simula contrafactuais acionáveis e traz o escore de papel.

**Na apresentação:** escolha "nunca fiz o exame" no colesterol. É onde o produto
demonstra, em um clique, a tese que o projeto inteiro sustenta.
